# 1. What you'll learn

        - separate trend, seasonality, and residual
- evaluate chronologically rather than randomly
- build a seasonal-naive baseline

        **The one question this notebook answers:** What does trend-seasonal decomposition reveal, and why must a seasonal-naive forecast be beaten?

# 2. Setup

This lesson keeps time-series decomposition and baselines small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_absolute_error,mean_squared_error

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Eight years of monthly demand combine a rising trend, 12-month seasonality, a level intervention, and noise. The last year is the forecast test.

**Small example:** If monthly values repeat [10,12,15,11,9], a seasonal-naive forecast for the next five is exactly [10,12,15,11,9]; it is cheap and often hard to beat.

In [ ]:
rng=np.random.default_rng(SEED);n=96;t=np.arange(n);season=8*np.sin(2*np.pi*t/12)+3*np.cos(2*np.pi*t/6);trend=30+.18*t;intervention=np.where(t>=60,6,0);y=trend+season+intervention+rng.normal(0,2,n);dates=pd.date_range("2018-01-01",periods=n,freq="MS");series=pd.Series(y,index=dates,name="demand");print("Shape:",series.shape,"range:",series.index.min().date(),"to",series.index.max().date());print(series.head().round(2))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
decomp=seasonal_decompose(series,model="additive",period=12,extrapolate_trend="freq");fig,axes=plt.subplots(1,3,figsize=(14,3.7));series.plot(ax=axes[0]);axes[0].set_title("Observed demand");decomp.trend.plot(ax=axes[1],color="#176b66");axes[1].set_title("Estimated trend");decomp.seasonal.iloc[:24].plot(ax=axes[2],color="#d38b45");axes[2].set_title("Repeating seasonal component");plt.tight_layout();plt.show()

**Takeaway:** Demand mixes long-term growth with a repeated annual pattern.

**Takeaway:** The smooth trend includes a visible level change after the intervention.

**Takeaway:** Seasonality repeats every twelve observations and should inform the baseline.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
train=series.iloc[:-12];test=series.iloc[-12:];print("Chronological train/test:",train.shape,test.shape,"cutoff:",train.index[-1].date());seasonal_naive=pd.Series(train.iloc[-12:].to_numpy(),index=test.index)

# 6. Train

        Training turns the assumptions behind time-series decomposition and baselines into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Choose a seasonal period from process knowledge and plots.
2. Estimate a smooth trend using centred moving averages.
3. Average detrended values by seasonal position.
4. Define residual as observed minus trend minus seasonal components.

In [ ]:
fit=seasonal_decompose(train,model="additive",period=12,extrapolate_trend="freq");season_pattern=fit.seasonal.iloc[-12:].to_numpy();drift=(train.iloc[-1]-train.iloc[-13])/12;drift_seasonal=pd.Series(train.iloc[-12:].to_numpy()+12*drift,index=test.index);print("Last-year drift/month:",round(drift,3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
mae0=mean_absolute_error(test,seasonal_naive);mae1=mean_absolute_error(test,drift_seasonal);mean_base=np.repeat(train.mean(),12);print(f"Seasonal-naive MAE={mae0:.3f}; drift+seasonal={mae1:.3f}; mean baseline={mean_absolute_error(test,mean_base):.3f}")
fig,axes=plt.subplots(1,3,figsize=(14,3.7));series.iloc[-36:].plot(ax=axes[0],label="actual");seasonal_naive.plot(ax=axes[0],label="seasonal naive");axes[0].set_title("Last-year holdout");axes[0].legend()
axes[1].plot(test.index,test-seasonal_naive,marker="o");axes[1].axhline(0,ls="--",color="black");axes[1].set_title("Baseline errors")
axes[2].bar(["mean","seasonal naive","drift+seasonal"],[mean_absolute_error(test,mean_base),mae0,mae1],color=[".6","#176b66","#d38b45"]);axes[2].set_title("Holdout MAE");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Random splitting and season estimates using future years leak temporal information and make baselines look unfairly strong. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
rng=np.random.default_rng(SEED);random_test=rng.choice(n,size=12,replace=False);leaked_template=np.array([series[series.index.month==series.index[i].month].mean() for i in random_test]);honest_template=np.array([train[train.index.month==series.index[i].month].mean() for i in random_test]);target=series.iloc[random_test].to_numpy()
print(f"Leaked all-years month template MAE={mean_absolute_error(target,leaked_template):.3f}; train-only month template={mean_absolute_error(target,honest_template):.3f}")

**Use this when…** you need to understand components and establish a credible low-cost forecast baseline.

        **Don't use this when…** seasonality changes rapidly, timestamps are irregular, or decomposition is being mistaken for causal attribution.

        ## Try this

        1. Change the intervention magnitude and inspect residuals.
2. Use multiplicative decomposition after making seasonal amplitude grow.
3. Backtest at six different historical cutoffs.